# 02 — Find the Pool by Its Job

**Target:** Apache Spark 4.2.0; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
Which JVM threads belong to task execution, RPC, shuffle and background maintenance?

**Why this matters:** thread count alone does not identify a leak or useful parallelism. A thread dump gives names and stack traces that help map activity to a subsystem.

In [ ]:
from pyspark.sql import SparkSession, functions as F
import os, subprocess, time
spark = SparkSession.builder.master('local[2]').appName('thread-lab-02').getOrCreate()
print('Driver PID:', os.getpid(), 'Spark:', spark.version)

In [ ]:
# Trigger a small shuffle, then leave the app alive for inspection.
(spark.range(0, 200_000).repartition(8, F.col('id') % 11)
 .groupBy(F.col('id') % 13).count().collect())
time.sleep(3)

## Capture a dump
Run `jcmd -l` to locate the Spark JVM, then `jcmd <JAVA_PID> Thread.print -l`. Do not assume the Python PID is the Java PID. On Windows, use `jcmd.exe`; on Linux/macOS, use `jcmd`. Search thread names and inspect stack traces, comparing an idle moment with the shuffle.

**Expected:** different pools have distinct names and call stacks. Task-running threads should not be confused with Netty event loops or RPC dispatchers.

**Why:** thread pools serve separate queues and responsibilities. Spark configuration for shuffle/RPC pools is release-sensitive; inspect the Spark 4.2.0 configuration reference before tuning.

**Safety:** only inspect processes you own or are authorized to inspect. Stop the session after capturing the dump.

In [ ]:
spark.stop()